In [59]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
cifar_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), 
    transforms.Resize((28, 28)),                 
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))         
])

mnist_transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))])

train_cifar = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform)
test_cifar = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform)# false olunca test seti indiriliyor

train_mnist = datasets.MNIST(root='./data', train=True, download=True, transform=mnist_transform)
test_mnist = datasets.MNIST(root='./data', train=False, download=True, transform=mnist_transform)

combined_train_dataset = ConcatDataset([train_cifar, train_mnist])

Veri setleri hazırlanıyor...
Files already downloaded and verified
Files already downloaded and verified


In [68]:
BATCH_SIZE = 128
lr = 0.001
patience = 3

In [69]:
train_loader = DataLoader(combined_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

test_loader_cifar = DataLoader(test_cifar, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_mnist = DataLoader(test_mnist, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

**UnifiedCNN**, hem MNIST hem de CIFAR-10 verilerini işleyebilecek şekilde tasarlanmıştır.

**Mimarinin Temel Özellikleri:**
* **Giriş Katmanı:** Model, `1 kanal` (Grayscale) ve `28x28` boyutunda görüntüler alır. Bu standartlaştırma, farklı veri setlerini aynı ağa sokabilmemizi sağlar.
* **Öznitelik Çıkarımı (Backbone):** 3 adet ardışık Konvolüsyon Bloğu (Conv -> BatchNorm -> ReLU -> Pool) kullanılmıştır. Bu katmanlar, görüntünün "domain"ini (yani MNIST mi yoksa CIFAR mı olduğunu) ve içeriğini (hangi sınıf olduğunu) ayırt edecek görsel öznitelikleri öğrenir.
* **Bağlama Dayalı Sınıflandırma (Context-Dependent Classification):** En kritik kısım **Çıktı Katmanı (fc3)**'dır. Modelin sadece **10 adet çıktı nöronu** vardır.
    * Model, **0. Sınıf** çıktısını hem MNIST'teki "0 Rakamı" hem de CIFAR'daki "Uçak" için aktif etmeyi öğrenmelidir.
    * Bunu yapabilmesi için ağın, önceki katmanlarda görüntünün bağlamını (doku, karmaşıklık vb.) örtük olarak (implicitly) öğrenmesi gerekir.

In [70]:
class UnifiedCNN(nn.Module):
    def __init__(self, num_classes=10): # 10 sınıf var (cifar10)
        super(UnifiedCNN, self).__init__()
        
        #blok 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2) # Çıktı: 16x16
        self.dropout1 = nn.Dropout2d(p=0.25)

        #blok 2
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2) # Çıktı: 8x8
        self.dropout2 = nn.Dropout2d(p=0.25)

        #blok 3
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d(2, 2) # Çıktı: 4x4
        self.dropout3 = nn.Dropout2d(p=0.25)

        # Fully Connected Layers
        self.flatten_dim = 256 * 3 * 3 #3x3 boyutunda 256 kanal var
        
        self.fc1 = nn.Linear(self.flatten_dim, 1024)
        self.bn_fc1 = nn.BatchNorm1d(1024)
        self.dropout_fc1 = nn.Dropout(p=0.5)
        
        self.fc2 = nn.Linear(1024, 512)
        self.bn_fc2 = nn.BatchNorm1d(512)
        self.dropout_fc2 = nn.Dropout(p=0.5)
        
        # Çıktı katmanı: Sadece 10 sınıf var.
        # Model, girdinin domainine göre bu 10 sınıfı "yorumlamayı" öğrenmeli.
        self.fc3 = nn.Linear(512, num_classes)

    def forward(self, x):
        #blok 1
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.dropout1(x)

        #blok 2
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.dropout2(x)

        #blok 3
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.relu(self.bn6(self.conv6(x)))
        x = self.pool3(x)
        x = self.dropout3(x)

        # Flatten & Classifier
        x = x.view(-1, self.flatten_dim)
        
        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout_fc1(x)
        
        x = F.relu(self.bn_fc2(self.fc2(x)))
        x = self.dropout_fc2(x)
        
        x = self.fc3(x)
        return x

In [71]:
model = UnifiedCNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=patience)#, verbose=True

In [72]:
def evaluate(model, loader, dataset_name="Dataset"):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    avg_loss = running_loss / len(loader)
    print(f"   -> {dataset_name} Accuracy: {accuracy:.2f}% | Loss: {avg_loss:.4f}")
    return avg_loss, accuracy

**Eğitim Stratejisi:**
1.  **Karma Batch (Mixed Batches):** `train_loader`, MNIST ve CIFAR-10 verilerinin birleştirilmesiyle (`ConcatDataset`) oluşturulmuştur ve `shuffle=True` ile karıştırılmıştır.
    * Bu sayede model, bir iterasyonda el yazısı bir rakam görürken, hemen ardından doğal bir nesne (örn. araba) görebilir.
    * Bu durum, modelin "ezberlemesini" engeller ve her girdi için dinamik olarak bağlamı analiz etmeye zorlar.

2.  **Ayrı Değerlendirme (Domain-Specific Evaluation):**
    * Model eğitilirken veriler karışıktır, ancak başarıyı ölçmek için `evaluate` fonksiyonu MNIST ve CIFAR-10 test setleri üzerinde **ayrı ayrı** çalıştırılır.
    * Bu, projenin başarı kriterini ölçer: Model aynı anda her iki alanda da (Domain A ve Domain B) yüksek başarım gösterebiliyor mu?

3.  **Optimizasyon:** `Adam` optimizasyonu ve `ReduceLROnPlateau` kullanılarak, loss (hata) değeri düştükçe öğrenme hızı (learning rate) ayarlanır ve modelin daha hassas yakınsaması sağlanır.

In [ ]:
def train(epochs=15):
    print(f"\nEğitim Başlıyor... Hedef Epoch: {epochs}")
    print("-" * 60)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            loop.set_postfix(loss=loss.item())
            
        train_acc = 100 * correct / total
        train_loss = running_loss / len(train_loader)
        
        print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        
        #Her epoch sonunda domain bazlı performansları ayrı ayrı ölçüyoruz
        #Model ikisini de öğreniyor mu?
        val_loss_mnist, val_acc_mnist = evaluate(model, test_loader_mnist, "MNIST Test")
        val_loss_cifar, val_acc_cifar = evaluate(model, test_loader_cifar, "CIFAR Test")
        
        # Scheduler adım (Ortalama loss'a göre learning rate düşür)
        avg_val_loss = (val_loss_mnist + val_loss_cifar) / 2
        scheduler.step(avg_val_loss)
        print("-" * 60)

In [67]:
train(epochs=5)


Eğitim Başlıyor... Hedef Epoch: 5
------------------------------------------------------------


Epoch 1: Train Loss: 0.8272 | Train Acc: 70.83%
   -> MNIST Test Accuracy: 98.89% | Loss: 0.0323
   -> CIFAR Test Accuracy: 60.74% | Loss: 1.1125
------------------------------------------------------------


Epoch 2: Train Loss: 0.5449 | Train Acc: 81.16%
   -> MNIST Test Accuracy: 99.16% | Loss: 0.0254
   -> CIFAR Test Accuracy: 69.18% | Loss: 0.8891
------------------------------------------------------------


Epoch 3: Train Loss: 0.4629 | Train Acc: 84.20%
   -> MNIST Test Accuracy: 99.33% | Loss: 0.0204
   -> CIFAR Test Accuracy: 72.60% | Loss: 0.7838
------------------------------------------------------------


Epoch 4: Train Loss: 0.4094 | Train Acc: 85.97%
   -> MNIST Test Accuracy: 99.17% | Loss: 0.0241
   -> CIFAR Test Accuracy: 75.85% | Loss: 0.7111
------------------------------------------------------------


Epoch 5: Train Loss: 0.3681 | Train Acc: 87.37%
   -> MNIST Test Accuracy: 99.36% | Loss: 0.0198
   -> CIFAR Test Accuracy: 76.28% | Loss: 0.6948
------------------------------------------------------------
